In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *
from scipy.ndimage import gaussian_filter
from fitting import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/2024-11/fdt/*'))
#files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/*fdt*'))
files_fdt

['/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T001503_V202602220112_0451090501.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T031503_V202602220112_0451090502.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T061503_V202602220112_0451090503.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T091503_V202602220112_0451090504.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T121503_V202602220112_0451090505.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T151503_V202602220112_0451090506.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T181503_V202602220112_0451090507.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T211503_V202602220112_0451090508.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241110T001503_V202602220112_0451100501.fits.gz',
 

In [3]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/2024-11/hmi/*'))
#files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/*hmi*'))
files_hmi

['/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_031630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_061630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_091630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_121630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_151630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_181630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_211630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241110_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241110_031630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241110_061630_TAI.2.magnetogram.fits',
 '/home/ul

In [4]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xu, yu = s['xu'], s['yu']

In [5]:
file_hmi = files_hmi[0]
file_fdt = files_fdt[0]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1,
                     xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                     crota=header_fdt['CROTA'] + 0.25,
                     grid=crop_grid(xu, yu, header_fdt),
                     kind='bilinear')

data_fdt[np.isnan(data_hmi)] = np.nan

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi - data_fdt, vmin=-40, vmax=40)

plt.axis('off')
plt.tight_layout()

In [7]:
I1 = data_fdt
I2 = data_hmi

Ix, Iy = np.gradient((I1 + I2) / 2)
It = I2 - I1

A = np.array([[gaussian_filter(Ix ** 2, 1), gaussian_filter(Ix * Iy, 1)],
              [gaussian_filter(Ix * Iy, 1), gaussian_filter(Iy ** 2, 1)]])

b = np.array([[-gaussian_filter(Ix * It, 1)],
              [-gaussian_filter(Iy * It, 1)]])

det = np.linalg.det(np.transpose(A, (2,3,0,1)))

/home/ulyanov/miniconda3/envs/myenv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2431: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


In [8]:
plt.figure(figsize=(10,10))
plt.imshow(det > 1e4, 'gray')
plt.axis('off')

(np.float64(-0.5), np.float64(895.5), np.float64(895.5), np.float64(-0.5))

In [9]:
Vx, Vy = np.transpose(np.linalg.solve(np.transpose(A, (2,3,0,1)), np.transpose(b, (2,3,0,1))), (2,3,0,1))[:,0,...]

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(Vx * (det > 1e5), 'gray', vmin=-2, vmax=2)
plt.axis('off')

(np.float64(-0.5), np.float64(895.5), np.float64(895.5), np.float64(-0.5))

In [11]:
xt, yt = np.where(det > 1e4)

Vx_ = Vx[xt, yt]
Vy_ = Vy[xt, yt]

In [12]:
plt.figure(figsize=(10,10))
plt.scatter(yt, Vy_, s=0.5)

In [13]:
x0, y0 = header_fdt['PXBEG2'] - 1, header_fdt['PXBEG1'] - 1

qx, bx = polyfit2d(Vx_, xt + x0, yt + y0, degree=2, return_coefficients=True)
qy, by = polyfit2d(Vy_, xt + x0, yt + y0, degree=2, return_coefficients=True)

qx, bx, qy, by

(array([-3.95701621e-04, -5.03740802e-03, -3.43876536e-07,  5.53531769e-07,
         2.35744003e-06]),
 array([2.7948921]),
 array([ 1.69443762e-03,  1.39585849e-03,  2.59101669e-07, -2.59790802e-06,
         4.70223001e-07]),
 array([-1.34416098]))

In [14]:
xi, yi = np.mgrid[:2048,:2048]

dx = qx[0] * xi + qx[1] * yi + qx[2] * xi ** 2 + qx[3] * xi * yi + qx[4] * yi ** 2 + bx
dy = qy[0] * xi + qy[1] * yi + qy[2] * xi ** 2 + qy[3] * xi * yi + qy[4] * yi ** 2 + by

In [17]:
plt.figure(figsize=(10,10))
plt.imshow(dx)

In [82]:
#np.savez('residual_distortion.npz', dx=dx, dy=dy)

In [18]:
xu += dx
yu += dy

In [19]:
file_hmi = files_hmi[0]
file_fdt = files_fdt[0]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1,
                     xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                     crota=header_fdt['CROTA'] + 0.25,
                     grid=crop_grid(xu, yu, header_fdt),
                     kind='bilinear')

data_fdt[np.isnan(data_hmi)] = np.nan

In [20]:
X = rebin(data_hmi, 1)
Y = rebin(data_fdt, 1)

xx, yy = [0], [0]

for q in np.arange(2., 30.25, 0.5):
    w = np.sqrt(X ** 2 + Y ** 2)
    t = np.where(w < q ** 2)
    x, y = X[t], Y[t]
    w = 1#w[t]

    A = np.array([[np.mean(w * x ** 2), np.mean(w * x * y)],
                  [np.mean(w * x * y), np.mean(w * y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

print(u / v)


xx = np.array(xx)
yy = np.array(yy)

1.2128880764328154


In [21]:
plt.figure(figsize=(10,10))
plt.plot([-400,400], [-400,400], color='black', lw=0.5)
plt.scatter(X, Y, s=0.05)
plt.plot([-u,u],[-v,v], '--', color='black', lw=1.5)
#plt.plot(xx, yy, '--', color='black', lw=1)
plt.plot()

plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-400,400)
plt.ylim(-400,400)



plt.grid(True)
plt.tight_layout()

In [22]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt[300:700,300:700], 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()

In [23]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi[300:700,300:700], 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()